# NEXUS v2 pilot — Flexible-TADA vs LoRA

هشت اجرای تک‌سید روی Qwen2-0.5B و SST-2. این notebook نتیجهٔ ازپیش‌آماده ندارد.

**اول:** از `Runtime > Change runtime type` گزینهٔ `T4 GPU` را انتخاب کنید.
سپس سلول‌ها را به ترتیب اجرا کنید. نتایج و checkpointها در Drive ذخیره می‌شوند.
اجرای Colab رایگان پس از بستن مرورگر تضمین‌شده نیست.

**دانش ثابت:** SenticNet 6 در بستهٔ senticnet 1.6؛ این پایلوت بازاجرای عین نسخهٔ قبلی مقاله نیست.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUTPUT = Path('/content/drive/MyDrive/nexus-v2-pilot/qwen2_sst2_seed42_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
SESSION_HOURS = 2.0
print('Output:', OUTPUT)


## دریافت کد ثابت از گیت‌هاب
اولین commit استفاده‌شده در Drive ثبت می‌شود؛ اتصال بعدی همان نسخه را می‌گیرد.
برای آزمایش با کد یا تنظیمات جدید، نام پوشهٔ OUTPUT را تغییر بدهید.

اگر ریپو هنوز ساخته نشده، در سلول بعد `USE_LOCAL_ZIP=True` بگذارید و ZIP کامل کد را آپلود کنید.


In [ ]:
import subprocess
import shutil
import zipfile
USE_LOCAL_ZIP = False  # Set True if the GitHub repository has not been created yet.
REPO = 'https://github.com/sepehrmustafavi/nexusv2.git'
CHECKOUT = Path('/content/nexus-v2')
if USE_LOCAL_ZIP:
    if not (CHECKOUT / 'SOURCE_COMMIT.txt').exists():
        from google.colab import files
        print('Upload nexus-v2-pilot.zip (the complete code archive)')
        uploads = files.upload()
        archive = next(name for name in uploads if name.endswith('.zip'))
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                destination = (Path('/content') / entry.filename).resolve()
                assert destination.is_relative_to(Path('/content')), 'Unsafe archive path'
            z.extractall('/content')
    commit = (CHECKOUT / 'SOURCE_COMMIT.txt').read_text().strip()
else:
    if CHECKOUT.exists() and not (CHECKOUT / '.git').exists():
        raise RuntimeError('Local ZIP checkout exists. Keep USE_LOCAL_ZIP=True or start a fresh runtime.')
    if not (CHECKOUT / '.git').exists():
        subprocess.run(['git', 'clone', REPO, str(CHECKOUT)], check=True)
    commit_file = OUTPUT / 'source_commit.txt'
    if commit_file.exists():
        commit = commit_file.read_text().strip()
        subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', 'origin', commit], check=True)
        subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', commit], check=True)
    else:
        commit = subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip()
commit_file = OUTPUT / 'source_commit.txt'
if commit_file.exists() and commit_file.read_text().strip() != commit:
    raise RuntimeError('Source revision changed: use the original archive/commit or a new OUTPUT folder.')
commit_file.write_text(commit + '\n')
print('Frozen source revision:', commit)


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CHECKOUT / 'requirements.txt')], check=True)
# The CUDA-enabled torch supplied by Colab is preserved if compatible.


## بررسی محیط و تست کوتاه
این تست‌ها مدل تصادفی کوچک دارند؛ نتیجهٔ پژوهشی نیستند. اگر GPU شناسایی نشد، نوع runtime را عوض کنید.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
from senticnet.senticnet import SenticNet
sn = SenticNet()
assert len(sn.data) == 200000, 'Unexpected SenticNet resource; do not mix protocols'
print('SenticNet 6 entries:', len(sn.data), '| good polarity:', sn.polarity_value('good'))
del sn
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CHECKOUT, check=True)


## اجرا یا ادامهٔ پایلوت
هر بار اجرای این سلول، حداکثر حدود دو ساعت از صف را پردازش می‌کند؛ سپس checkpoint می‌گیرد و برمی‌گردد.
سلول را دوباره اجرا کنید تا پیام `Completed 8/8` نمایش داده شود.
اگر Colab قطع شد، مجدداً GPU بگیرید، Drive را mount کنید و از بالا اجرا کنید.
صددرصدِ هشت اجرا ممکن است در یک سهمیه یا جلسهٔ Colab رایگان تمام نشود.


In [ ]:
command = [sys.executable, '-u', '-m', 'nexus_pilot.run',
           '--output', str(OUTPUT), '--max-hours', str(SESSION_HOURS)]
subprocess.run(command, cwd=CHECKOUT, check=True)


## ارزیابی نهایی — فقط وقتی هر هشت اجرا تمام شده‌اند
تنظیمات با tune داخلی انتخاب می‌شوند؛ این سلول نتیجهٔ چهار حالت منتخب را روی validation رسمی گزارش می‌کند.
اگر صف کامل نباشد، عمداً خطا می‌دهد. برای ادامه به سلول قبل برگردید.


In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'nexus_pilot.run',
                '--output', str(OUTPUT), '--finalize'], cwd=CHECKOUT, check=True)


In [ ]:
import pandas as pd
import json
from IPython.display import display
from google.colab import files
display(pd.read_csv(OUTPUT / 'summary.csv'))
print(json.dumps(json.loads((OUTPUT / 'effects.json').read_text()), indent=2))
files.download(str(OUTPUT / 'summary.csv'))


## چه چیزهایی برای تحلیل برگردانیم؟
`summary.csv`، `effects.json`، `protocol.json` و در صورت نیاز `all_candidates.json` و فایل‌های `predictions.csv`.
فایل‌های بزرگ checkpoint را لازم نیست ارسال کنید.
نتایج تک‌سید برای انتخاب مسیر اولیه‌اند؛ برتری قطعی یا توضیح‌پذیری را اثبات نمی‌کنند.
